In [1]:
# Husayn El Sharif

## Overview

The SOCO region covers Georgia, Alabama, and parts of Mississippi.

This script downloads hourly weather data for major cities and climate regions in the SOCO area from the Open-Meteo API and saves it to a CSV file for further analysis.

## Data Source: Open-Meteo
- Free, no API key required  
- Hourly weather data available since 1940  
- Uses ERA5 reanalysis (~25 km grid resolution)  

## Environment
Use conda environment: `energy_demand_ml_env001`

In [2]:
# imports
import time
import requests
import pandas as pd
import os
from pathlib import Path


In [3]:
# Define SOCO region locations (latitude, longitude) 
locations = {
    "atlanta_ga": (33.7490, -84.3880),
    "savannah_ga": (32.0809, -81.0912),
    "albany_ga": (31.5785, -84.1557),
    "birmingham_al": (33.5186, -86.8104),
    "mobile_al": (30.6954, -88.0399),
    "huntsville_al": (34.7304, -86.5861),
    "meridian_ms": (32.3643, -88.7037),
}

In [4]:
# Open-Meteo API get_weather function
def _get_weather_chunk(
    lat,
    lon,
    start_date,
    end_date,
    timeout=60,
    max_retries=10,
    pause_seconds=15,
):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "dew_point_2m",
            "precipitation",
            "surface_pressure",
            "wind_speed_10m",
            "shortwave_radiation",
        ],
        "timezone": "UTC",
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"Requesting lat={lat}, lon={lon}, "
                f"{start_date} -> {end_date}, attempt {attempt}/{max_retries}"
            )

            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()

            data = response.json()

            if "hourly" not in data or "time" not in data["hourly"]:
                raise ValueError(f"Unexpected API response: {data}")

            df = pd.DataFrame(data["hourly"])
            df["datetime_utc"] = pd.to_datetime(df["time"], utc=True)
            df = df.drop(columns=["time"])

            return df

        except Exception as e:
            last_error = e
            print(f"  Failed: {e}")

            if attempt < max_retries:
                sleep_time = pause_seconds * attempt
                print(f"  Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    raise RuntimeError(
        f"Failed for lat={lat}, lon={lon}, {start_date} -> {end_date}. "
        f"Last error: {last_error}"
    )


def get_weather(
    lat,
    lon,
    start="2015-01-01",
    end="2026-03-01",
    chunk_freq="YS",   # yearly chunks
):
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)

    # Build chunk boundaries
    boundaries = list(pd.date_range(start=start_ts, end=end_ts, freq=chunk_freq))

    if not boundaries or boundaries[0] != start_ts:
        boundaries = [start_ts] + boundaries

    if boundaries[-1] != end_ts:
        boundaries.append(end_ts)

    chunks = []

    for i in range(len(boundaries) - 1):
        chunk_start = boundaries[i]
        chunk_end = boundaries[i + 1] - pd.Timedelta(days=1)

        if i == len(boundaries) - 2:
            chunk_end = end_ts

        df_chunk = _get_weather_chunk(
            lat=lat,
            lon=lon,
            start_date=chunk_start.strftime("%Y-%m-%d"),
            end_date=chunk_end.strftime("%Y-%m-%d"),
        )
        chunks.append(df_chunk)

    df = pd.concat(chunks, ignore_index=True)
    df = df.drop_duplicates(subset=["datetime_utc"]).sort_values("datetime_utc")
    df = df.reset_index(drop=True)

    return df

In [5]:
# Pull data from Open-Meteo for each location and combine into a single DataFrame

weather_dfs = []

for name, (lat, lon) in locations.items():
    try:
        df_loc = get_weather(lat, lon, start="2015-01-01", end="2026-03-01")
        df_loc["location"] = name
        weather_dfs.append(df_loc)
        print(f"Finished {name}: {len(df_loc):,} rows")
    except Exception as e:
        print(f"Skipped {name} due to error: {e}")

weather = pd.concat(weather_dfs, ignore_index=True)

Requesting lat=33.749, lon=-84.388, 2015-01-01 -> 2015-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2016-01-01 -> 2016-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2017-01-01 -> 2017-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2018-01-01 -> 2018-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2019-01-01 -> 2019-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2020-01-01 -> 2020-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2021-01-01 -> 2021-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2022-01-01 -> 2022-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2023-01-01 -> 2023-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2024-01-01 -> 2024-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2025-01-01 -> 2025-12-31, attempt 1/10
Requesting lat=33.749, lon=-84.388, 2026-01-01 -> 2026-03-01, attempt 1/10
Finished atlanta_ga: 97,872 rows
Requesting lat=32.0809, lon=-81.0912, 2015-01-01 -> 2015-12-31, att

In [6]:
weather_dfs

[       temperature_2m  relative_humidity_2m  dew_point_2m  precipitation  \
 0                 5.5                    67          -0.2            0.0   
 1                 4.4                    66          -1.3            0.0   
 2                 3.5                    66          -2.3            0.0   
 3                 2.7                    66          -3.0            0.0   
 4                 2.0                    67          -3.4            0.0   
 ...               ...                   ...           ...            ...   
 97867            23.2                    29           4.4            0.0   
 97868            23.4                    29           4.4            0.0   
 97869            23.5                    30           4.8            0.0   
 97870            23.4                    31           5.3            0.0   
 97871            20.3                    51           9.9            0.0   
 
        surface_pressure  wind_speed_10m  shortwave_radiation  \
 0       

In [7]:
weather

,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,wind_speed_10m,shortwave_radiation,datetime_utc,location
0,5.5,67,-0.2,0.0,988.9,7.4,0.0,2015-01-01 00:00:00+00:00,atlanta_ga
1,4.4,66,-1.3,0.0,988.9,7.3,0.0,2015-01-01 01:00:00+00:00,atlanta_ga
2,3.5,66,-2.3,0.0,988.7,6.0,0.0,2015-01-01 02:00:00+00:00,atlanta_ga
3,2.7,66,-3.0,0.0,989.0,6.0,0.0,2015-01-01 03:00:00+00:00,atlanta_ga
4,2.0,67,-3.4,0.0,988.7,6.6,0.0,2015-01-01 04:00:00+00:00,atlanta_ga
...,...,...,...,...,...,...,...,...,...
685099,24.4,35,8.1,0.0,1008.4,4.7,744.0,2026-03-01 19:00:00+00:00,meridian_ms
685100,24.9,34,7.9,0.0,1007.7,3.6,725.0,2026-03-01 20:00:00+00:00,meridian_ms
685101,25.0,33,7.4,0.0,1007.2,4.5,585.0,2026-03-01 21:00:00+00:00,meridian_ms
685102,24.2,35,7.7,0.0,1007.0,5.9,355.0,2026-03-01 22:00:00+00:00,meridian_ms


In [8]:
# Prvot to wide table with one row per datetime and columns for each location's weather data

weather_wide = weather.pivot(
    index="datetime_utc",
    columns="location"
)

# flatten columns
weather_wide.columns = [
    f"{var}_{loc}" for var, loc in weather_wide.columns
]

# reset index to make datetime_utc a column again
weather_wide = weather_wide.reset_index()

In [9]:
weather_wide

,datetime_utc,temperature_2m_albany_ga,temperature_2m_atlanta_ga,temperature_2m_birmingham_al,temperature_2m_huntsville_al,temperature_2m_meridian_ms,temperature_2m_mobile_al,temperature_2m_savannah_ga,relative_humidity_2m_albany_ga,relative_humidity_2m_atlanta_ga,...,wind_speed_10m_meridian_ms,wind_speed_10m_mobile_al,wind_speed_10m_savannah_ga,shortwave_radiation_albany_ga,shortwave_radiation_atlanta_ga,shortwave_radiation_birmingham_al,shortwave_radiation_huntsville_al,shortwave_radiation_meridian_ms,shortwave_radiation_mobile_al,shortwave_radiation_savannah_ga
0,2015-01-01 00:00:00+00:00,9.2,5.5,4.8,1.8,5.8,10.9,8.7,76,67,...,10.9,13.5,9.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-01-01 01:00:00+00:00,8.2,4.4,3.7,0.9,4.8,10.1,7.9,80,66,...,10.5,13.8,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2015-01-01 02:00:00+00:00,7.3,3.5,2.8,0.2,3.9,9.8,7.3,83,66,...,10.4,13.3,12.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2015-01-01 03:00:00+00:00,6.5,2.7,2.0,-0.4,3.5,9.5,6.7,86,66,...,10.2,13.3,13.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015-01-01 04:00:00+00:00,5.7,2.0,1.3,-0.9,3.2,9.3,6.4,89,67,...,9.7,13.4,13.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97867,2026-03-01 19:00:00+00:00,22.8,23.2,23.0,23.2,24.4,23.0,21.1,45,29,...,4.7,4.6,3.9,799.0,719.0,642.0,707.0,744.0,806.0,780.0
97868,2026-03-01 20:00:00+00:00,23.5,23.4,23.4,23.4,24.9,23.3,21.6,40,29,...,3.6,6.3,5.5,716.0,605.0,656.0,580.0,725.0,748.0,685.0
97869,2026-03-01 21:00:00+00:00,23.6,23.5,23.5,23.6,25.0,22.0,21.1,36,30,...,4.5,12.1,7.9,568.0,541.0,542.0,554.0,585.0,610.0,532.0
97870,2026-03-01 22:00:00+00:00,23.1,23.4,23.1,23.0,24.2,20.6,19.4,36,31,...,5.9,13.4,8.3,382.0,284.0,310.0,371.0,355.0,376.0,338.0


In [10]:
# list of columns in weather_wide
weather_wide.columns.tolist()

['datetime_utc',
 'temperature_2m_albany_ga',
 'temperature_2m_atlanta_ga',
 'temperature_2m_birmingham_al',
 'temperature_2m_huntsville_al',
 'temperature_2m_meridian_ms',
 'temperature_2m_mobile_al',
 'temperature_2m_savannah_ga',
 'relative_humidity_2m_albany_ga',
 'relative_humidity_2m_atlanta_ga',
 'relative_humidity_2m_birmingham_al',
 'relative_humidity_2m_huntsville_al',
 'relative_humidity_2m_meridian_ms',
 'relative_humidity_2m_mobile_al',
 'relative_humidity_2m_savannah_ga',
 'dew_point_2m_albany_ga',
 'dew_point_2m_atlanta_ga',
 'dew_point_2m_birmingham_al',
 'dew_point_2m_huntsville_al',
 'dew_point_2m_meridian_ms',
 'dew_point_2m_mobile_al',
 'dew_point_2m_savannah_ga',
 'precipitation_albany_ga',
 'precipitation_atlanta_ga',
 'precipitation_birmingham_al',
 'precipitation_huntsville_al',
 'precipitation_meridian_ms',
 'precipitation_mobile_al',
 'precipitation_savannah_ga',
 'surface_pressure_albany_ga',
 'surface_pressure_atlanta_ga',
 'surface_pressure_birmingham_al',


## Weather Data Dictionary (SOCO Region - Wide Format)

| Column | Description |
|--------|-------------|
| datetime_utc | Timestamp in Coordinated Universal Time (UTC) at hourly resolution. |

### Temperature (°C)
| Column | Description |
|--------|-------------|
| temperature_2m_albany_ga | Air temperature at 2 meters above ground in Albany, GA. |
| temperature_2m_atlanta_ga | Air temperature at 2 meters above ground in Atlanta, GA. |
| temperature_2m_birmingham_al | Air temperature at 2 meters above ground in Birmingham, AL. |
| temperature_2m_huntsville_al | Air temperature at 2 meters above ground in Huntsville, AL. |
| temperature_2m_meridian_ms | Air temperature at 2 meters above ground in Meridian, MS. |
| temperature_2m_mobile_al | Air temperature at 2 meters above ground in Mobile, AL. |
| temperature_2m_savannah_ga | Air temperature at 2 meters above ground in Savannah, GA. |

### Relative Humidity (%)
| Column | Description |
|--------|-------------|
| relative_humidity_2m_albany_ga | Relative humidity at 2 meters in Albany, GA (%). |
| relative_humidity_2m_atlanta_ga | Relative humidity at 2 meters in Atlanta, GA (%). |
| relative_humidity_2m_birmingham_al | Relative humidity at 2 meters in Birmingham, AL (%). |
| relative_humidity_2m_huntsville_al | Relative humidity at 2 meters in Huntsville, AL (%). |
| relative_humidity_2m_meridian_ms | Relative humidity at 2 meters in Meridian, MS (%). |
| relative_humidity_2m_mobile_al | Relative humidity at 2 meters in Mobile, AL (%). |
| relative_humidity_2m_savannah_ga | Relative humidity at 2 meters in Savannah, GA (%). |

### Dew Point (°C)
| Column | Description |
|--------|-------------|
| dew_point_2m_albany_ga | Dew point temperature at 2 meters in Albany, GA. |
| dew_point_2m_atlanta_ga | Dew point temperature at 2 meters in Atlanta, GA. |
| dew_point_2m_birmingham_al | Dew point temperature at 2 meters in Birmingham, AL. |
| dew_point_2m_huntsville_al | Dew point temperature at 2 meters in Huntsville, AL. |
| dew_point_2m_meridian_ms | Dew point temperature at 2 meters in Meridian, MS. |
| dew_point_2m_mobile_al | Dew point temperature at 2 meters in Mobile, AL. |
| dew_point_2m_savannah_ga | Dew point temperature at 2 meters in Savannah, GA. |

### Precipitation (mm)
| Column | Description |
|--------|-------------|
| precipitation_albany_ga | Total precipitation during the hour in Albany, GA (mm). |
| precipitation_atlanta_ga | Total precipitation during the hour in Atlanta, GA (mm). |
| precipitation_birmingham_al | Total precipitation during the hour in Birmingham, AL (mm). |
| precipitation_huntsville_al | Total precipitation during the hour in Huntsville, AL (mm). |
| precipitation_meridian_ms | Total precipitation during the hour in Meridian, MS (mm). |
| precipitation_mobile_al | Total precipitation during the hour in Mobile, AL (mm). |
| precipitation_savannah_ga | Total precipitation during the hour in Savannah, GA (mm). |

### Surface Pressure (hPa)
| Column | Description |
|--------|-------------|
| surface_pressure_albany_ga | Surface atmospheric pressure in Albany, GA (hPa). |
| surface_pressure_atlanta_ga | Surface atmospheric pressure in Atlanta, GA (hPa). |
| surface_pressure_birmingham_al | Surface atmospheric pressure in Birmingham, AL (hPa). |
| surface_pressure_huntsville_al | Surface atmospheric pressure in Huntsville, AL (hPa). |
| surface_pressure_meridian_ms | Surface atmospheric pressure in Meridian, MS (hPa). |
| surface_pressure_mobile_al | Surface atmospheric pressure in Mobile, AL (hPa). |
| surface_pressure_savannah_ga | Surface atmospheric pressure in Savannah, GA (hPa). |

### Wind Speed (m/s)
| Column | Description |
|--------|-------------|
| wind_speed_10m_albany_ga | Wind speed at 10 meters above ground in Albany, GA (m/s). |
| wind_speed_10m_atlanta_ga | Wind speed at 10 meters above ground in Atlanta, GA (m/s). |
| wind_speed_10m_birmingham_al | Wind speed at 10 meters above ground in Birmingham, AL (m/s). |
| wind_speed_10m_huntsville_al | Wind speed at 10 meters above ground in Huntsville, AL (m/s). |
| wind_speed_10m_meridian_ms | Wind speed at 10 meters above ground in Meridian, MS (m/s). |
| wind_speed_10m_mobile_al | Wind speed at 10 meters above ground in Mobile, AL (m/s). |
| wind_speed_10m_savannah_ga | Wind speed at 10 meters above ground in Savannah, GA (m/s). |

### Shortwave Radiation (W/m²)
| Column | Description |
|--------|-------------|
| shortwave_radiation_albany_ga | Incoming shortwave solar radiation in Albany, GA (W/m²). |
| shortwave_radiation_atlanta_ga | Incoming shortwave solar radiation in Atlanta, GA (W/m²). |
| shortwave_radiation_birmingham_al | Incoming shortwave solar radiation in Birmingham, AL (W/m²). |
| shortwave_radiation_huntsville_al | Incoming shortwave solar radiation in Huntsville, AL (W/m²). |
| shortwave_radiation_meridian_ms | Incoming shortwave solar radiation in Meridian, MS (W/m²). |
| shortwave_radiation_mobile_al | Incoming shortwave solar radiation in Mobile, AL (W/m²). |
| shortwave_radiation_savannah_ga | Incoming shortwave solar radiation in Savannah, GA (W/m²). |

In [11]:
# export to CSV to "data" folder
# make sure to create "data" folder if it doesn't exist

output_dir = Path("data")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

soco_region_hourly_weather_csv_path = output_dir / "soco_region_hourly_weather.csv"    
weather_wide.to_csv(soco_region_hourly_weather_csv_path, index=False)